# Analysis: Do LLMs Know When They're Being Unoriginal?

Correlating internal model signals with human-rated originality.

**Hypotheses:**
1. Low self-perplexity → humans rate output as less original
2. High token entropy → humans rate output as more original
3. N-gram novelty and semantic novelty dissociate (surface ≠ meaning)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import krippendorff
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Load data

In [ ]:
signals = pd.read_csv('../signals/signal_matrix.csv')
ratings_raw = pd.read_csv('../data/human_ratings.csv')  # columns: output_id, rater_id, rating

print('Signals shape:', signals.shape)
print('Ratings shape:', ratings_raw.shape)
signals.head()

## 2. Inter-rater agreement (Krippendorff's α)

In [ ]:
# Pivot to rater × item matrix
rating_matrix = ratings_raw.pivot(index='rater_id', columns='output_id', values='rating').values
alpha = krippendorff.alpha(reliability_data=rating_matrix, level_of_measurement='ordinal')
print(f"Krippendorff's α = {alpha:.3f}")
print('Interpretation: α > 0.6 = acceptable, α > 0.8 = good')

## 3. Compute mean human originality score per output

In [ ]:
mean_ratings = ratings_raw.groupby('output_id')['rating'].mean().reset_index()
mean_ratings.columns = ['output_id', 'human_originality']

# Merge with signal matrix (match on run_id as output_id)
signals['output_id'] = signals.index
df = signals.merge(mean_ratings, on='output_id', how='inner')
print(f'Merged dataframe: {df.shape}')
df.describe()

## 4. Correlation: each signal vs human originality

In [ ]:
signal_cols = ['mean_token_entropy', 'self_perplexity', 'ngram_novelty', 'type_token_ratio', 'centroid_distance']

results = []
for sig in signal_cols:
    r, p = stats.spearmanr(df[sig], df['human_originality'])
    results.append({'signal': sig, 'spearman_r': round(r, 3), 'p_value': round(p, 4), 'significant': p < 0.05})

results_df = pd.DataFrame(results).sort_values('spearman_r', ascending=False)
print(results_df.to_string(index=False))

## 5. Scatter plots: signals vs human originality

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

colors = {'gpt2-xl': '#4C72B0', 'mistral-7b': '#DD8452'}

for i, sig in enumerate(signal_cols):
    ax = axes[i]
    for model, grp in df.groupby('model'):
        ax.scatter(grp[sig], grp['human_originality'],
                   alpha=0.4, s=20, label=model, color=colors.get(model, 'gray'))
    r, p = stats.spearmanr(df[sig], df['human_originality'])
    ax.set_xlabel(sig.replace('_', ' ').title(), fontsize=10)
    ax.set_ylabel('Human Originality (1-5)', fontsize=10)
    ax.set_title(f'ρ = {r:.3f}, p = {p:.3f}', fontsize=11)
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Internal Signals vs Human-Rated Originality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../signals/correlation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to signals/correlation_plots.png')

## 6. Test hypothesis: does self-perplexity predict low originality?

In [ ]:
# Split into low/high self-perplexity groups (median split)
median_pp = df['self_perplexity'].median()
low_pp  = df[df['self_perplexity'] <= median_pp]['human_originality']
high_pp = df[df['self_perplexity'] >  median_pp]['human_originality']

t_stat, p_val = stats.mannwhitneyu(low_pp, high_pp, alternative='less')
print(f'Low self-perplexity mean originality:  {low_pp.mean():.3f}')
print(f'High self-perplexity mean originality: {high_pp.mean():.3f}')
print(f'Mann-Whitney U p-value (one-tailed): {p_val:.4f}')
print()
if p_val < 0.05:
    print('RESULT: Hypothesis SUPPORTED — low self-perplexity outputs rated less original.')
else:
    print('RESULT: Hypothesis NOT supported at p<0.05.')

## 7. Per-model breakdown

In [ ]:
for model, grp in df.groupby('model'):
    print(f'\n=== {model} ===')
    for sig in signal_cols:
        r, p = stats.spearmanr(grp[sig], grp['human_originality'])
        print(f'  {sig:<30} ρ={r:+.3f}  p={p:.3f}')

## 8. Summary table — export for write-up

In [ ]:
summary = results_df.copy()
summary.to_csv('../signals/summary_table.csv', index=False)
print('Summary saved to signals/summary_table.csv')
summary